# 02 · LoRA의 행렬 구조

Low-Rank Adaptation(LoRA, 저차원 행렬을 이용한 모델 적응)의 원리를 작은 행렬로 관찰합니다. 실제 Adapter 학습이 아니라 특이값 분해(Singular Value Decomposition, SVD)로 변화량을 근사합니다. 약 10분.

모델 다운로드·API 키·GPU 없이 실행합니다. 합성 자료만 사용하며, 실제 LLM 학습·성능 검증과 구분합니다. 예측 → 실행 → 변수 변경 → 근거 기록의 순서로 진행합니다.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from course_labs.adaptation import *
OUT = ROOT / 'output' / 'validation' / 'adaptation-labs'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__)
assert sys.version_info[:2] == (3, 11), '공통 환경 Python 3.11 커널을 선택하세요.'


## 고정 가중치와 변화량

기존 가중치 W와 적용할 변화량 ΔW를 분리합니다.

In [ ]:
rng = np.random.default_rng(42)
W = rng.normal(size=(12, 8)); delta = rng.normal(size=(12, 8))
W_original = W.copy()
print('W/ΔW:', W.shape, delta.shape)

## rank 2의 행렬

B는 12×2, A는 2×8이고 BA는 12×8입니다.

In [ ]:
r2 = rank_approximation(delta, 2)
print('B:', r2['b'].shape, 'A:', r2['a'].shape, 'BA:', r2['update'].shape)
assert r2['update'].shape == W.shape

## 학습 대상 수의 비교

작은 예시에서는 rank가 커지면 오히려 전체 행렬보다 파라미터가 많아질 수 있습니다.

In [ ]:
rows = [{'rank': r, 'full': 96, 'factor': rank_approximation(delta, r)['factor_params']} for r in (1,2,4,8)]
display(rows)

## 근사 오차

Frobenius norm 기준 상대 오차입니다. 언어 모델 품질 점수가 아닙니다.

In [ ]:
metrics = [{'rank': r, 'relative_error': rank_approximation(delta,r)['relative_error']} for r in (1,2,4,8)]
display(metrics)
assert metrics[-1]['relative_error'] < 1e-12

## rank와 오차의 시각화

길이가 짧을수록 행렬 근사 오차가 작은 것입니다. 업무 성능과 혼동하지 마세요.

In [ ]:
display(HTML(''.join(f"<p>rank {m['rank']} <span style='display:inline-block;background:#284b72;height:14px;width:{m['relative_error']*400}px'></span> {m['relative_error']:.3f}</p>" for m in metrics)))

## 저차원 구조의 예시

본래 rank 2인 변화량은 rank 2로 거의 완벽하게 복원할 수 있습니다.

In [ ]:
structured = rng.normal(size=(12,2)) @ rng.normal(size=(2,8))
print(rank_approximation(structured,2)['relative_error'])
assert rank_approximation(structured,2)['relative_error'] < 1e-12

## 일반 LoRA의 scaling

설명용 α/r scaling입니다. 라이브러리의 다른 scaling 방식은 구별해야 합니다.

In [ ]:
alpha, rank = 16, 2
adapted = W + (alpha/rank) * r2['update']
assert np.array_equal(W, W_original)
print('원본 고정:', np.array_equal(W,W_original))

## 입력에 대한 출력 변화

행렬의 출력 변화를 직접 계산합니다. 텍스트 생성 결과가 아닙니다.

In [ ]:
x = rng.normal(size=8)
y_base, y_adapted = W @ x, adapted @ x
print('출력 차이 L2:', np.linalg.norm(y_adapted-y_base))

## 큰 층의 비용 계산

4096×4096 선형 층 하나만 비교합니다. 전체 모델 메모리 추정이 아닙니다.

In [ ]:
d = 4096
for r in (8,16,32):
    print(r, 'full', d*d, 'factors', 2*d*r, '비율', (2*d*r)/(d*d))

## 자원 추정의 한계

가중치 외에 gradient, optimizer state, activation, 임시 버퍼가 필요합니다.

In [ ]:
weight_bytes = 4096*4096*2
print('FP16 가중치만 MiB:', weight_bytes/1024**2)
print('GPU 최대 메모리 측정값 아님')

## 실험 기록

원본 가중치 고정과 오차 정의를 산출물에 함께 기록합니다.

In [ ]:
save_report(OUT / 'low-rank-report.json', {'method':'SVD approximation; not trained LoRA', 'shape': list(delta.shape), 'metrics': metrics, 'parameters': rows})

## 실행 기록과 기획서 연결

1. 바꾼 변수와 달라진 결과
2. 이 실험으로 확인하지 못한 부분
3. 프로젝트에서 추가로 검증할 가정

출력은 `output/validation/adaptation-labs/`에 저장됩니다. 개인정보나 학생 제출물은 공개 저장소에 넣지 않습니다.